# Sequential plus self-play training

Train one model at the second-to-last turn, then initialize and run self-play from that checkpoint. `initial_training_steps` controls the number of optimizer iterations used for this model.

The notebook keeps training data and physics caches under `curling_data/`; model checkpoints are saved under `weights/<category>/<run>/checkpoints/`, with records under `stats/`.


In [ ]:
import importlib
from dataclasses import asdict
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import sequential_training as seq
import self_play_training as self_play
from bot import ThrowPerturbations
importlib.reload(seq)
importlib.reload(self_play)


In [ ]:
# Shared experiment settings
N = 5
seed = 0
grid_size = (10, 10, 4)
robust_top_fraction = 0.05
perturbation_angle_offsets_deg = (0.0, -0.2, 0.2)
perturbation_probabilities = (0.5, 0.25, 0.25)
perturbations = ThrowPerturbations(
    angle_offsets_deg=perturbation_angle_offsets_deg,
    probabilities=perturbation_probabilities,
)

# Train one initial model for the second-to-last turn (m_9 for N=5).
rows_per_model = 20_000
initial_training_steps = 30
random_throw_count = 36 * 4 * 3
policy_comparison_num_sims = 5
state_generator_version = 'turn_based'
data_root = Path('/home/vietafan/curling_data')
# Reuse the prior artifact location when present; its old name included the stage count.
sequential_run_dir = data_root / f'N{N}_rows{rows_per_model}_round{2 * N - 1}'
legacy_sequential_run_dir = data_root / f'N{N}_rows{rows_per_model}_stages2'
if not sequential_run_dir.exists() and legacy_sequential_run_dir.exists():
    sequential_run_dir = legacy_sequential_run_dir
sequential_artifact_dir = sequential_run_dir / 'sequential_experiment'
sequential_diagnostic_output_dir = None  # Set to a path to save diagnostic checkpoints.
sequential_run_name = f'round{2 * N - 1}_{rows_per_model // 1000}k'
sequential_weights_dir = seq.training_run_weights_dir(
    'sequential_training', sequential_run_name
)
initial_model_path = sequential_weights_dir / 'checkpoints' / f'm_{2 * N - 1}.npz'
initial_dataset_path = sequential_artifact_dir / f'D_{2 * N - 1}.npz'

# Self-play settings
num_iterations = 10
num_states_per_iteration = 200
halflife = 5.0
hidden_layer_size = 20
batch_size = 100
training_iterations = 10
learning_rate = 0.5
resume_from = None  # Set to a saved iteration to resume this self-play run.
self_play_run_name = 'round9_2k_v2'
self_play_artifact_dir = data_root / f'self_play_N{N}_{self_play_run_name}'
self_play_weights_dir = seq.training_run_weights_dir(
    'self_play', self_play_run_name
)

config = self_play.SelfPlayConfig(
    num_states=num_states_per_iteration,
    num_iterations=num_iterations,
    grid_size=grid_size,
    state_generator_version=state_generator_version,
    halflife=halflife,
    max_stones=2 * N,
    num_stones_per_side=N,
    hidden_layer_size=hidden_layer_size,
    batch_size=batch_size,
    training_iterations=training_iterations,
    learning_rate=learning_rate,
    robust_top_fraction=robust_top_fraction,
    perturbations=perturbations,
)
print(f'self-play retention probability: {config.retention_probability():.4f}')
run_parameters = {
    'N': N,
    'seed': seed,
    'initial_sequential_training': {
        'run_name': sequential_run_name,
        'turn': 2 * N - 1,
        'training_stages': 1,
        'resume_from': None,
        'rows_per_model': rows_per_model,
        'num_training_iterations': initial_training_steps,
        'state_generator_version': state_generator_version,
        'grid_size': list(grid_size),
        'random_throw_count': random_throw_count,
        'robust_top_fraction': robust_top_fraction,
        'perturbations': asdict(perturbations),
        'policy_comparison_num_sims': policy_comparison_num_sims,
        'training_data_generation': {
            'method': 'Sample second-to-last-turn boards, select the final throw, and label each board with the resulting team-0 net score.',
            'rows_per_model': rows_per_model,
        },
        'network_training': {
            'hidden_layer_size': 20,
            'batch_size': 100,
            'loss': 'softmax cross-entropy',
            'optimizer': 'mini-batch gradient descent',
            'regularization': 0.0,
            'learning_rate_schedule': '0.5 * (1 + cos(pi * iteration / num_training_iterations))',
            'validation_fraction': 0.2,
            'batch_shuffle_seed': 'model index + iteration',
        },
    },
    'self_play': {
        'run_name': self_play_run_name,
        'config': asdict(config),
        'resume_from': resume_from,
    },
    'artifacts': {
        'data_root': data_root.name,
        'initial_model': str(initial_model_path.relative_to(repo_root)),
        'initial_dataset': str(initial_dataset_path.relative_to(data_root)),
        'sequential_artifact_dir': str(sequential_artifact_dir.relative_to(data_root)),
        'sequential_weights_dir': str(sequential_weights_dir.relative_to(repo_root)),
        'self_play_artifact_dir': str(self_play_artifact_dir.relative_to(data_root)),
        'self_play_weights_dir': str(self_play_weights_dir.relative_to(repo_root)),
        'diagnostic_output_dir': (str(sequential_diagnostic_output_dir) if sequential_diagnostic_output_dir else None),
    },
}


In [ ]:
sequential_experiment = seq.make_experiment_setup(
    state_generator_version,
    seed=seed,
    grid_size=grid_size,
    perturbations=perturbations,
    random_throw_count=random_throw_count,
    robust_top_fraction=robust_top_fraction,
)
initial_run = seq.train_sequential_models(
    N=N,
    rows_per_model=rows_per_model,
    artifact_dir=sequential_artifact_dir,
    resume_from=None,
    training_stages=1,
    num_training_iterations=initial_training_steps,
    experiment=sequential_experiment,
    diagnostic_output_dir=sequential_diagnostic_output_dir,
    policy_comparison_num_sims=policy_comparison_num_sims,
    weights_dir=sequential_weights_dir,
)
assert 2 * N - 1 in initial_run.models
initial_model_path = sequential_weights_dir / 'checkpoints' / f'm_{2 * N - 1}.npz'
initial_dataset_path = sequential_artifact_dir / f'D_{2 * N - 1}.npz'
print(f'self-play starts from {initial_model_path}')


In [ ]:
results = self_play.run_self_play(
    config=config,
    artifact_dir=self_play_artifact_dir,
    initial_model_path=initial_model_path,
    initial_dataset_path=initial_dataset_path,
    seed=seed,
    resume_from=resume_from,
    weights_dir=self_play_weights_dir,
    notebook_parameters=run_parameters,
)
results

In [ ]:
# Each iteration writes D_i.npz and X_i.npz under curling_data, and m_i.npz under weights/<category>/<run>/checkpoints/.
for i, record in results.items():
    print(i, record['new_rows'], record['training_rows'], record['loss'][-1])